[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/081.%20The%203D%20Container_Truck%20Loading%20Problem%20%28Bin%20Packing%29/P81-Tier-1_executed.ipynb)

---


# 81. The 3D Container/Truck Loading Problem

## Tier 1: Grid-Based Greedy Packing

### Purpose and scope

This notebook studies how to place rectangular items inside a rectangular container while respecting container boundaries, non-overlap, a weight limit, and a simplified support requirement. It reports the resulting volume utilization and the packed items' center of gravity.

The implementation uses a one-unit position grid and a greedy placement rule: items are considered in descending volume order by default, and each item is placed at the first feasible position and orientation found. This is a constructive heuristic, not a mixed-integer programming solver and not a complete search over all packings. It can produce a feasible packing, but it does not certify that the packing is optimal.

### Modeling assumptions

- Each item and the container are axis-aligned rectangular boxes with positive integer dimensions, expressed in consistent length units.
- An item may use any distinct permutation of its three dimensions as its orientation. These are the possible axis-aligned rotations; symmetric items can have fewer than six distinct orientations.
- Item positions are restricted to nonnegative integer coordinates. Consequently, feasible placements that require fractional coordinates are outside this grid model.
- Items may touch at faces, edges, or corners, but their interiors may not overlap.
- A non-floor item is considered supported when the total area of its bottom face overlapping items directly beneath it is at least 50% of its bottom-face area. This is a geometric support proxy, not a full mechanical stability or load-bearing analysis.
- The weight limit applies to the total weight of packed items. Weight distribution, item strength, center-of-gravity limits, unloading order, and container-floor load limits are not constrained.

### Solution sequence

1. Represent items, the container, and a packing solution.
2. Define tests for overlap, container boundaries, and the support-area rule.
3. Scan candidate grid positions and orientations for each item in the chosen order.
4. Report the resulting packing and verify its modeled constraints.
5. Examine how container dimensions and item order affect this heuristic's result.

### Example instance

The container has dimensions $10 \times 8 \times 6$ units and a weight limit of 1000 kg. The items are:

| Item | Dimensions (length × width × height) | Weight |
|---|---:|---:|
| 1 | $4 \times 3 \times 2$ units | 10 kg |
| 2 | $3 \times 3 \times 3$ units | 15 kg |
| 3 | $2 \times 4 \times 3$ units | 8 kg |

The total item volume divided by container volume is an upper bound on volume utilization if every item is packed. It is not, by itself, proof that all items can be packed together.

## Mathematical model and the placement rule

### Feasible packing conditions

Let $I$ be the set of items. For each item $i \in I$, let $(l_i,w_i,h_i)$ be its original dimensions, $v_i=l_iw_ih_i$ its volume, and $q_i$ its weight. Let the container dimensions be $(L,W,H)$ and its weight limit be $Q$. A placement assigns item $i$ an integer-coordinate position $(x_i,y_i,z_i)$ and an orientation $(\ell_i,\omega_i,\eta_i)$, which is a permutation of $(l_i,w_i,h_i)$.

A packing is feasible under this notebook's model when each packed item is within the container, each pair of packed items is non-overlapping, the total packed weight is within the limit, and every item above the floor meets the support-area threshold. The boundary conditions are:

$$
0 \leq x_i,\quad 0 \leq y_i,\quad 0 \leq z_i,\quad
x_i+\ell_i \leq L,\quad y_i+\omega_i \leq W,\quad z_i+\eta_i \leq H.
$$

For two items $i$ and $j$, their interiors do not overlap when at least one of these separations holds:

$$
x_i+\ell_i \leq x_j\ \lor\ x_j+\ell_j \leq x_i\ \lor\
y_i+\omega_i \leq y_j\ \lor\ y_j+\omega_j \leq y_i\ \lor\
z_i+\eta_i \leq z_j\ \lor\ z_j+\eta_j \leq z_i.
$$

The weight condition is $\sum_{i\text{ packed}} q_i \leq Q$. For an item above the floor, let $A_i$ be the sum of the bottom-face areas that overlap items whose top faces are at the item's bottom height. The support condition used here is $A_i \geq 0.5\ell_i\omega_i$.

### Objective and heuristic

The packing objective is to maximize the total volume of packed items:

$$
\max \sum_{i\text{ packed}} v_i.
$$

This objective describes the packing goal; the code does not solve this optimization problem exactly. Instead, it applies a greedy constructive rule. It scans the grid

$$
\mathcal{G}=\{(x,y,z): x\in\{0,\ldots,L-1\},\ y\in\{0,\ldots,W-1\},\ z\in\{0,\ldots,H-1\}\}
$$

in nested $x,y,z$ order, and scans each item's distinct orientations in a deterministic order. It accepts the first candidate satisfying the boundary, non-overlap, weight, and support tests, then moves to the next item. An item for which no candidate is accepted is left unpacked.

If $n=|I|$, $g=|\mathcal{G}|$, and each item has at most six orientations, a straightforward upper bound on the placement checks is proportional to $6gn^2$: each candidate can be checked against up to $n$ already placed items. This describes the grid-scanning implementation, not the complexity of an exact optimization algorithm. The greedy result can depend on item order and does not establish global optimality.

In the code, `check_within_container`, `check_overlap`, and `check_stability` implement the respective feasibility tests. `solve_greedy_3d_packing` performs the grid scan, and `volume_utilization` reports packed volume divided by container volume.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations
from matplotlib.patches import Patch


class Item:
    """Represent a rectangular item with dimensions, weight, and identifier."""

    def __init__(self, length, width, height, weight, item_id):
        self.length = length
        self.width = width
        self.height = height
        self.weight = weight
        self.item_id = item_id

    def volume(self):
        """Return the item's volume in cubic length units."""
        return self.length * self.width * self.height

    def get_orientations(self):
        """Return distinct axis-aligned orientations in deterministic order."""
        dimensions = (self.length, self.width, self.height)
        return sorted(set(permutations(dimensions)))

    def __repr__(self):
        return (
            f"Item{self.item_id}({self.length}×{self.width}×{self.height}, "
            f"{self.weight} kg)"
        )


class Container:
    """Represent a rectangular container and its total weight limit."""

    def __init__(self, length, width, height, max_weight=None):
        self.length = length
        self.width = width
        self.height = height
        self.max_weight = (
            float("inf") if max_weight is None else max_weight
        )

    def volume(self):
        """Return the container's volume in cubic length units."""
        return self.length * self.width * self.height

    def __repr__(self):
        return (
            f"Container({self.length}×{self.width}×{self.height}, "
            f"max_weight={self.max_weight} kg)"
        )


class PackingSolution:
    """Store item placements and calculate packing summary measures."""

    def __init__(self):
        self.placements = []

    def add_item(self, item, position, orientation):
        """Record an item's lower-corner position and oriented dimensions."""
        self.placements.append(
            {
                "item": item,
                "position": position,
                "orientation": orientation,
            }
        )

    def total_volume(self):
        """Return the sum of the volumes of packed items."""
        return sum(placement["item"].volume() for placement in self.placements)

    def total_weight(self):
        """Return the sum of the weights of packed items."""
        return sum(placement["item"].weight for placement in self.placements)

    def volume_utilization(self, container):
        """Return packed volume as a percentage of container volume."""
        if container.volume() <= 0:
            raise ValueError("Container dimensions must have positive volume.")
        return self.total_volume() / container.volume() * 100

    def center_of_gravity(self):
        """Return the weight-weighted center of packed items."""
        if not self.placements:
            return None

        total_weight = self.total_weight()
        if total_weight <= 0:
            raise ValueError("Packed items must have positive total weight.")

        weighted_x = 0.0
        weighted_y = 0.0
        weighted_z = 0.0

        for placement in self.placements:
            item = placement["item"]
            x, y, z = placement["position"]
            length, width, height = placement["orientation"]

            # Each rectangular item's center is its lower corner plus half its dimensions.
            item_center_x = x + length / 2
            item_center_y = y + width / 2
            item_center_z = z + height / 2

            weighted_x += item_center_x * item.weight
            weighted_y += item_center_y * item.weight
            weighted_z += item_center_z * item.weight

        return (
            weighted_x / total_weight,
            weighted_y / total_weight,
            weighted_z / total_weight,
        )

In [ ]:
def check_overlap(item1_pos, item1_orient, item2_pos, item2_orient):
    """Return whether two axis-aligned items overlap in their interiors."""
    x1, y1, z1 = item1_pos
    length1, width1, height1 = item1_orient
    x2, y2, z2 = item2_pos
    length2, width2, height2 = item2_orient

    # Touching at a face, edge, or corner is allowed; overlap must occur on all axes.
    overlap_x = not (x1 + length1 <= x2 or x2 + length2 <= x1)
    overlap_y = not (y1 + width1 <= y2 or y2 + width2 <= y1)
    overlap_z = not (z1 + height1 <= z2 or z2 + height2 <= z1)

    return overlap_x and overlap_y and overlap_z


def check_within_container(position, orientation, container):
    """Return whether an item's full extent lies inside the container."""
    x, y, z = position
    length, width, height = orientation

    return (
        x >= 0
        and y >= 0
        and z >= 0
        and x + length <= container.length
        and y + width <= container.width
        and z + height <= container.height
    )


def check_stability(placement, existing_placements):
    """Apply the model's 50% bottom-face support-area test."""
    position = placement["position"]
    orientation = placement["orientation"]
    x, y, z = position
    length, width, _ = orientation

    # The container floor supports every point of an item placed at height zero.
    if z == 0:
        return True

    support_area = 0
    bottom_area = length * width

    for existing in existing_placements:
        existing_x, existing_y, existing_z = existing["position"]
        existing_length, existing_width, existing_height = existing["orientation"]

        # Only items whose top face meets this item's bottom face provide support.
        if z == existing_z + existing_height:
            overlap_x = max(
                0,
                min(x + length, existing_x + existing_length)
                - max(x, existing_x),
            )
            overlap_y = max(
                0,
                min(y + width, existing_y + existing_width)
                - max(y, existing_y),
            )
            support_area += overlap_x * overlap_y

    return support_area >= 0.5 * bottom_area

In [ ]:
def solve_greedy_3d_packing(items, container, sort_items=True):
    """Place items using a one-unit grid scan and a first-feasible rule."""
    dimensions = (container.length, container.width, container.height)
    if any(dimension <= 0 or int(dimension) != dimension for dimension in dimensions):
        raise ValueError("Container dimensions must be positive integers for the unit grid.")
    if container.max_weight < 0:
        raise ValueError("The container weight limit cannot be negative.")
    for item in items:
        item_dimensions = (item.length, item.width, item.height)
        if any(
            dimension <= 0 or int(dimension) != dimension
            for dimension in item_dimensions
        ):
            raise ValueError("Item dimensions must be positive integers for the unit grid.")
        if item.weight < 0:
            raise ValueError("Item weights cannot be negative.")

    print(f"Solving 3D packing for {len(items)} items using a grid-based greedy rule...")

    positions = [
        (x, y, z)
        for x in range(int(container.length))
        for y in range(int(container.width))
        for z in range(int(container.height))
    ]

    items_to_place = list(items)
    if sort_items:
        items_to_place.sort(key=lambda item: item.volume(), reverse=True)

    current_solution = PackingSolution()

    for item in items_to_place:
        if current_solution.total_weight() + item.weight > container.max_weight:
            print(f"  Skipped {item}: adding it would exceed the weight limit.")
            continue

        best_placement = None

        for orientation in item.get_orientations():
            for position in positions:
                if not check_within_container(position, orientation, container):
                    continue

                has_overlap = any(
                    check_overlap(
                        position,
                        orientation,
                        existing["position"],
                        existing["orientation"],
                    )
                    for existing in current_solution.placements
                )
                if has_overlap:
                    continue

                test_placement = {
                    "item": item,
                    "position": position,
                    "orientation": orientation,
                }
                if not check_stability(test_placement, current_solution.placements):
                    continue

                best_placement = (position, orientation)
                break

            if best_placement is not None:
                break

        if best_placement is None:
            print(f"  Could not place {item} under the current grid and feasibility rules.")
            continue

        position, orientation = best_placement
        current_solution.add_item(item, position, orientation)
        print(f"  Packed {item} at {position} with orientation {orientation}")

    return current_solution

In [ ]:
def visualize_3d_packing(solution, container):
    """Plot the container frame and the packed axis-aligned items."""
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection="3d")

    corners = [
        (0, 0, 0),
        (container.length, 0, 0),
        (container.length, container.width, 0),
        (0, container.width, 0),
        (0, 0, container.height),
        (container.length, 0, container.height),
        (container.length, container.width, container.height),
        (0, container.width, container.height),
    ]
    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]

    for start, end in edges:
        edge_points = [corners[start], corners[end]]
        ax.plot3D(*zip(*edge_points), color="black", alpha=0.45, linewidth=1.5)

    colors = ["tab:red", "tab:blue", "tab:green", "tab:orange", "tab:purple"]
    legend_handles = []

    for index, placement in enumerate(solution.placements):
        item = placement["item"]
        x, y, z = placement["position"]
        length, width, height = placement["orientation"]
        color = colors[index % len(colors)]

        ax.bar3d(
            x,
            y,
            z,
            length,
            width,
            height,
            color=color,
            alpha=0.65,
            edgecolor="black",
            shade=True,
        )
        ax.text(
            x + length / 2,
            y + width / 2,
            z + height / 2,
            f"Item {item.item_id}",
            fontsize=8,
            ha="center",
            va="center",
        )
        legend_handles.append(Patch(facecolor=color, label=f"Item {item.item_id}"))

    ax.set_xlabel("Length (units)")
    ax.set_ylabel("Width (units)")
    ax.set_zlabel("Height (units)")
    ax.set_title(
        "3D Container Packing\n"
        f"Volume utilization: {solution.volume_utilization(container):.1f}%"
    )
    ax.set_xlim(0, container.length)
    ax.set_ylim(0, container.width)
    ax.set_zlim(0, container.height)
    ax.set_box_aspect((container.length, container.width, container.height))
    if legend_handles:
        ax.legend(handles=legend_handles, title="Packed items")

    plt.tight_layout()
    plt.show()


def print_solution_summary(solution, container):
    """Print packed-item details and summary measures."""
    print("\n" + "=" * 60)
    print("3D CONTAINER LOADING SOLUTION SUMMARY")
    print("=" * 60)
    print(f"\nContainer: {container}")
    print(f"Container volume: {container.volume():.1f} cubic units")
    print(f"\nItems packed: {len(solution.placements)}")
    print(f"Total packed volume: {solution.total_volume():.1f} cubic units")
    print(f"Total packed weight: {solution.total_weight():.1f} kg")
    print(f"Volume utilization: {solution.volume_utilization(container):.1f}%")

    center_of_gravity = solution.center_of_gravity()
    if center_of_gravity is None:
        print("Center of gravity: not defined because no items are packed")
    else:
        print(
            "Center of gravity (length, width, height): "
            f"({center_of_gravity[0]:.2f}, "
            f"{center_of_gravity[1]:.2f}, "
            f"{center_of_gravity[2]:.2f}) units"
        )

    print("\nItem details:")
    print("-" * 50)
    for placement in solution.placements:
        item = placement["item"]
        print(f"{item}:")
        print(f"  Lower-corner position: {placement['position']} units")
        print(f"  Oriented dimensions: {placement['orientation']} units")
        print(f"  Volume: {item.volume():.1f} cubic units")
        print(f"  Weight: {item.weight} kg")
        print()

In [ ]:
# Define the example container and item data used throughout the notebook.
container = Container(length=10, width=8, height=6, max_weight=1000)

items = [
    Item(4, 3, 2, 10, 1),
    Item(3, 3, 3, 15, 2),
    Item(2, 4, 3, 8, 3),
]

total_item_volume = sum(item.volume() for item in items)
total_item_weight = sum(item.weight for item in items)

print("3D CONTAINER LOADING PROBLEM")
print("=" * 40)
print(f"Container: {container}")
print("\nItems to pack:")
for item in items:
    print(f"  {item}")

print(f"\nTotal item volume: {total_item_volume:.1f} cubic units")
print(f"Total item weight: {total_item_weight} kg")
print(
    "Volume-ratio upper bound if all items are packed: "
    f"{total_item_volume / container.volume() * 100:.1f}%"
)

## Construct a packing

The solver below applies the stated greedy rule to the example data. By default, it orders items by descending volume, then scans each item's orientations and grid positions until it finds the first feasible placement. Each placement is checked against the items already packed. The returned solution is the result of this particular order and scan rule; it is not a proof of optimality.

In [ ]:
solution = solve_greedy_3d_packing(items, container)
print_solution_summary(solution, container)

## Visualize the packing

Each colored cuboid represents one packed item, and the wireframe shows the container boundaries. The legend identifies items by color. The axes use the same length units as the input dimensions.

In [ ]:
visualize_3d_packing(solution, container)

## Verify the modeled constraints

A solution is useful only if its reported placements satisfy the conditions enforced by the model. This cell independently counts boundary, pairwise overlap, weight, and support-rule violations. The support check follows the placement order, so an item can receive support only from items already placed beneath it.

In [ ]:
print("\n" + "=" * 60)
print("PACKING CONSTRAINT VERIFICATION")
print("=" * 60)

boundary_violations = sum(
    not check_within_container(
        placement["position"], placement["orientation"], container
    )
    for placement in solution.placements
)

overlap_count = 0
for first_index, first in enumerate(solution.placements):
    for second in solution.placements[first_index + 1:]:
        if check_overlap(
            first["position"],
            first["orientation"],
            second["position"],
            second["orientation"],
        ):
            overlap_count += 1

weight_constraint_ok = solution.total_weight() <= container.max_weight
stability_violations = sum(
    not check_stability(placement, solution.placements[:index])
    for index, placement in enumerate(solution.placements)
)

all_constraints_satisfied = (
    boundary_violations == 0
    and overlap_count == 0
    and weight_constraint_ok
    and stability_violations == 0
)

print(f"Boundary constraint violations: {boundary_violations}")
print(f"Pairwise overlap violations: {overlap_count}")
print(f"Weight constraint satisfied: {weight_constraint_ok}")
print(
    f"Total weight: {solution.total_weight()} kg / "
    f"Limit: {container.max_weight} kg"
)
print(f"Support-rule violations: {stability_violations}")
print(f"All modeled constraints satisfied: {all_constraints_satisfied}")

assert boundary_violations == 0
assert overlap_count == 0
assert weight_constraint_ok
assert stability_violations == 0
assert solution.total_volume() == sum(
    placement["item"].volume() for placement in solution.placements
)

## Sensitivity analysis

A greedy packing can respond to instance dimensions and item order. The first experiment changes the container dimensions while keeping the items fixed. The second experiment passes items in several orders and disables the solver's default volume sorting, so each tested order is actually used. These experiments illustrate behavior of the specified heuristic; they do not establish an optimality guarantee.

In [ ]:
print("\n" + "=" * 60)
print("SENSITIVITY ANALYSIS")
print("=" * 60)

print("\nEffect of container dimensions on utilization:")
print("-" * 45)

size_variations = [
    (8, 6, 4),
    (10, 8, 6),
    (12, 10, 8),
]

for length, width, height in size_variations:
    test_container = Container(length, width, height)
    test_solution = solve_greedy_3d_packing(items, test_container)
    utilization = test_solution.volume_utilization(test_container)
    print(
        f"Container {length}×{width}×{height}: "
        f"{utilization:.1f}% utilization, "
        f"{len(test_solution.placements)} items packed"
    )

print("\nEffect of item order on the heuristic result:")
print("-" * 50)

ordering_strategies = [
    ("Largest volume first", lambda item: item.volume(), True),
    ("Smallest volume first", lambda item: item.volume(), False),
    ("Heaviest first", lambda item: item.weight, True),
    ("Lightest first", lambda item: item.weight, False),
]

for strategy_name, key_function, reverse in ordering_strategies:
    ordered_items = sorted(items, key=key_function, reverse=reverse)
    test_solution = solve_greedy_3d_packing(
        ordered_items,
        container,
        sort_items=False,
    )
    print(
        f"{strategy_name}: "
        f"{test_solution.volume_utilization(container):.1f}% utilization, "
        f"{len(test_solution.placements)} items packed"
    )

## Interpretation and limitations

Volume utilization is the total volume of packed items divided by the container volume. It measures space use, but it does not describe how easy the load is to unload or whether weight is safely distributed. The reported center of gravity is the weight-weighted average of the centers of the packed boxes; it is a descriptive quantity, not a stability constraint in this model.

The grid scan and first-feasible placement rule make the construction transparent and suitable for small demonstrations. The result can depend on item order, orientation order, and grid resolution. In particular, a feasible arrangement may be missed when it requires non-integer coordinates, and a locally acceptable placement may prevent later items from fitting. The verification cell checks the constraints represented here, including the simplified 50% support-area rule; it does not validate the omitted physical considerations.

For the example, the total item volume divided by the container volume gives the utilization that would result if all three items were packed. Whether the heuristic reaches that bound is determined by its placements and should be read from the executed summary, rather than assumed from the volume calculation alone.